# Autonomous ML Experiment Investigator

A stateful ML methodology agent that investigates an unfamiliar supervised-learning dataset, gathers evidence through constrained tools, proposes a baseline experiment, requests human approval, executes deterministic scikit-learn code, critiques the methodology, and revises the plan when necessary.

```text
Task
 ↓
diagnostic agent
 ↓
read-only tools
 ↓
structured plan
 ↓
human approval
 ↓
deterministic experiment
 ↓
methodology critic
 ├─ accept → report
 └─ revise → approval → experiment
```

The central comparison is **schema-only LLM reasoning vs. a tool-using LangGraph agent**.

## 1. Design principles

The model controls diagnosis, planning, critique, and reporting. Python controls data access, splitting, preprocessing, training, and metrics.

The LLM never receives arbitrary Python or shell execution. Tools are primarily read-only observations, and model training occurs only after a structured plan and an external approval boundary.

## 2. Environment

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Annotated, Any, Literal, TypedDict
import json
import operator
import os
import re
import uuid

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.datasets import make_classification, make_regression
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, ToolRuntime
from langgraph.types import Command, interrupt

load_dotenv()

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

DATA_DIR = Path('data') / 'agent_benchmark'
ARTIFACT_DIR = Path('artifacts')
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'gpt-5.6-luna'
MAX_EXPERIMENT_ITERATIONS = 3

print('Imports ready.')

Imports ready.


In [2]:
if not os.getenv('OPENAI_API_KEY'):
    from getpass import getpass
    os.environ['OPENAI_API_KEY'] = getpass('OpenAI API key: ')

llm = ChatOpenAI(
    model=MODEL_NAME,
    reasoning_effort='medium',
    use_responses_api=True,
)

print('Model:', MODEL_NAME)

Model: gpt-5.6-luna


## 3. Controlled benchmark datasets

In [3]:
def generate_imbalanced_churn():
    X, y = make_classification(
        n_samples=3500,
        n_features=8,
        n_informative=5,
        n_redundant=1,
        weights=[0.97, 0.03],
        class_sep=1.0,
        random_state=RANDOM_STATE,
    )
    df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(X.shape[1])])
    df['tenure_months'] = rng.integers(1, 72, len(df))
    df['region'] = rng.choice(['north', 'south', 'east', 'west'], size=len(df))
    df['customer_id'] = [f'CUST_{i:05d}' for i in range(len(df))]
    df['churn'] = y
    return df


def generate_temporal_demand():
    n = 3000
    timestamp = pd.date_range('2024-01-01', periods=n, freq='h')
    hour = timestamp.hour.to_numpy()
    day = np.arange(n) / 24
    temperature = 22 + 6 * np.sin(2 * np.pi * day / 30) + rng.normal(0, 2, n)
    promo = rng.binomial(1, 0.08, n)
    demand = (
        100
        + 0.025 * np.arange(n)
        + 18 * np.sin(2 * np.pi * hour / 24)
        + 2.5 * temperature
        + 28 * promo
        + rng.normal(0, 8, n)
    )
    return pd.DataFrame({
        'timestamp': timestamp,
        'temperature': temperature,
        'promo': promo,
        'hour': hour,
        'demand': demand,
    })


def generate_grouped_failure():
    n_machines = 450
    observations_per_machine = 6
    n = n_machines * observations_per_machine
    machine_num = np.repeat(np.arange(n_machines), observations_per_machine)
    latent_machine_risk = rng.normal(0, 1.0, n_machines)[machine_num]
    vibration = rng.normal(0.5 + 0.4 * latent_machine_risk, 0.7, n)
    temperature = rng.normal(65 + 4 * latent_machine_risk, 6, n)
    load = rng.uniform(0.2, 1.0, n)
    logit = -2.1 + 1.0 * latent_machine_risk + 0.7 * vibration + 0.012 * (temperature - 65) + 0.6 * load
    probability = 1 / (1 + np.exp(-logit))
    failure = rng.binomial(1, probability)
    return pd.DataFrame({
        'machine_id': [f'M_{x:04d}' for x in machine_num],
        'vibration': vibration,
        'temperature': temperature,
        'load': load,
        'failure': failure,
    })


def generate_leaky_default():
    X, y = make_classification(
        n_samples=3200,
        n_features=6,
        n_informative=4,
        n_redundant=1,
        weights=[0.82, 0.18],
        class_sep=0.8,
        random_state=RANDOM_STATE + 1,
    )
    df = pd.DataFrame(X, columns=[
        'credit_score_signal', 'debt_ratio_signal', 'income_signal',
        'utilization_signal', 'history_signal', 'noise_signal',
    ])

    # Deliberate post-outcome variables.
    df['default_status_after_30d'] = y
    df['collection_amount_after_default'] = y * rng.uniform(500, 5000, len(df))
    df['application_id'] = [f'APP_{i:05d}' for i in range(len(df))]
    df['defaulted'] = y
    return df


def generate_iid_regression():
    X, y = make_regression(
        n_samples=2800,
        n_features=8,
        n_informative=6,
        noise=18,
        random_state=RANDOM_STATE,
    )
    df = pd.DataFrame(X, columns=[f'x_{i}' for i in range(X.shape[1])])
    df['segment'] = rng.choice(['A', 'B', 'C'], size=len(df))
    df['target'] = y
    return df

In [4]:
DATASETS = {
    "dataset_01": {
        "target": "churn",
        "problem_type": "classification",
        "objective": (
            "Predict customer churn for future unseen customers. "
            "The positive class is the business event of interest."
        ),
        "generator": generate_imbalanced_churn,
    },

    "dataset_02": {
        "target": "demand",
        "problem_type": "regression",
        "objective": (
            "Predict demand for future timestamps using information "
            "available before the forecast period."
        ),
        "generator": generate_temporal_demand,
    },

    "dataset_03": {
        "target": "failure",
        "problem_type": "classification",
        "objective": (
            "Predict failure for previously unseen machines. "
            "Several historical rows can belong to the same machine."
        ),
        "generator": generate_grouped_failure,
    },

    "dataset_04": {
        "target": "defaulted",
        "problem_type": "classification",
        "objective": (
            "Predict loan default at application time. "
            "Only information available at application time is valid."
        ),
        "generator": generate_leaky_default,
    },

    "dataset_05": {
        "target": "target",
        "problem_type": "regression",
        "objective": (
            "Predict a continuous outcome for IID observations."
        ),
        "generator": generate_iid_regression,
    },
}


for dataset_id, spec in DATASETS.items():
    path = DATA_DIR / f"{dataset_id}.csv"

    if not path.exists():
        spec["generator"]().to_csv(path, index=False)

    print(dataset_id, pd.read_csv(path).shape)

dataset_01 (3500, 12)
dataset_02 (3000, 5)
dataset_03 (2700, 5)
dataset_04 (3200, 10)
dataset_05 (2800, 10)


In [5]:
def load_dataset(dataset_id: str) -> pd.DataFrame:
    if dataset_id not in DATASETS:
        raise KeyError(f'Unknown dataset_id: {dataset_id}')
    df = pd.read_csv(DATA_DIR / f'{dataset_id}.csv')
    for column in df.columns:
        if 'time' in column.lower() or 'date' in column.lower():
            try:
                df[column] = pd.to_datetime(df[column])
            except Exception:
                pass
    return df


def get_task_spec(dataset_id: str) -> dict[str, Any]:
    spec = DATASETS[dataset_id]
    return {
        'dataset_id': dataset_id,
        'target': spec['target'],
        'problem_type': spec['problem_type'],
        'objective': spec['objective'],
    }

Five synthetic tasks encode known methodology requirements:

| Dataset | Methodological issue |
|---|---|
| `dataset_01` | severe class imbalance |
| `dataset_02` | chronological generalization |
| `dataset_03` | repeated machine entities |
| `dataset_04` | post-outcome leakage |
| `dataset_05` | ordinary IID regression |

Synthetic data gives controlled ground truth while still forcing realistic validation decisions.

## 4. Explicit graph state

In [6]:
class AgentState(TypedDict, total=False):
    dataset_id: str
    target: str
    problem_type: str
    objective: str

    messages: Annotated[list[Any], add_messages]

    plan: dict[str, Any]
    human_approved: bool
    human_note: str

    experiments: Annotated[list[dict[str, Any]], operator.add]
    critique: dict[str, Any]
    experiment_iteration: int

    final_report: str

## 5. Diagnostic tools

In [7]:
@tool
def inspect_schema(runtime: ToolRuntime) -> str:
    """Inspect dataset shape, columns, dtypes, unique counts, and a small sample."""

    def extract_column_info(column: pd.Series):
        return {
            "dtype": str(column.dtype),
            "unique_count": int(column.nunique(dropna=False)),
        }

    dataset_id = runtime.state["dataset_id"]

    df = load_dataset(dataset_id)

    columns_info = {
        column_name: extract_column_info(column)
        for column_name, column in df.items()
    }

    sample = df.head(5).to_dict(orient="records")

    return json.dumps({
        "shape": {
            "rows": df.shape[0],
            "columns": df.shape[1],
        },
        "columns_info": columns_info,
        "sample": sample,
    }, default=str)


@tool
def profile_target(runtime: ToolRuntime) -> str:

    """Profile the prediction target, including class balance or regression summary."""

    dataset_id = runtime.state["dataset_id"]
    target = runtime.state["target"]
    problem_type = runtime.state["problem_type"]

    df = load_dataset(dataset_id)
    n = len(df)

    if problem_type == 'classification':
        
        target_profile = (
            df[target].value_counts(dropna=False).
            reset_index().
            assign(
            proportion = lambda X_df: X_df['count']/n
            ).to_dict(orient='records')
        )

    elif problem_type == 'regression':
        target_profile = (
            df[target].
            describe().
            to_dict()
        )

    else:
        raise RuntimeError("Invalid problem type.")


    return json.dumps({
        'target_column' : target,
        'problem_type' : problem_type,
        'target_profile' : target_profile
    })
    

@tool
def inspect_missingness(runtime: ToolRuntime) -> str:
    """Inspect missing-value counts and rates for every column."""

    dataset_id = runtime.state["dataset_id"]

    df = load_dataset(dataset_id)
    n = len(df)

    missingness = [
        {
            "column": column,
            "missing_count": int(df[column].isna().sum()),
            "missing_rate": float(df[column].isna().sum() / n),
        }
        for column in df.columns
    ]

    return json.dumps({
        "missingness": missingness
    })


@tool
def inspect_duplicates(runtime: ToolRuntime) -> str:
    """Count exact duplicate rows."""

    dataset_id = runtime.state["dataset_id"]

    df = load_dataset(dataset_id)

    duplicate_count = int(df.duplicated().sum())

    return json.dumps({
        "duplicate_count": duplicate_count,
        "duplicate_rate": duplicate_count / len(df),
    })

In [8]:
@tool
def inspect_time_candidate(column: str, runtime: ToolRuntime) -> str:
    """Inspect a candidate time column: parseability, range, ordering, and target behavior across time."""

    dataset_id = runtime.state["dataset_id"]
    target = runtime.state["target"]

    df = load_dataset(dataset_id)

    if column not in df.columns:
        return json.dumps({"error": f"Column '{column}' not found."})

    parsed = pd.to_datetime(df[column], errors="coerce")
    non_missing = df[column].notna().sum()

    valid_df = (
        df.loc[parsed.notna(), [target]]
        .assign(_time=parsed[parsed.notna()])
        .sort_values("_time")
    )

    # Divide chronology into four roughly equal periods
    periods = np.array_split(valid_df, 4)

    target_over_time = [
        {
            "period": i + 1,
            "start": period["_time"].min(),
            "end": period["_time"].max(),
            "count": len(period),
            "target_mean": float(period[target].mean()),
        }
        for i, period in enumerate(periods)
        if len(period) > 0
    ]

    return json.dumps({
        "column": column,
        "parseable_rate": (
            float(parsed.notna().sum() / non_missing)
            if non_missing else 0.0
        ),
        "min_time": parsed.min(),
        "max_time": parsed.max(),
        "is_monotonic_increasing": bool(
            parsed.dropna().is_monotonic_increasing
        ),
        "target_over_time": target_over_time,
    }, default=str)


@tool
def inspect_group_candidate(column: str, runtime: ToolRuntime) -> str:
    """Inspect whether a column identifies repeated entities that could leak across random train/validation splits."""

    dataset_id = runtime.state["dataset_id"]

    df = load_dataset(dataset_id)

    if column not in df.columns:
        return json.dumps({"error": f"Column '{column}' not found."})

    counts = df[column].value_counts(dropna=False)

    repeated_groups = counts[counts > 1]

    return json.dumps({
        "column": column,
        "row_count": len(df),
        "unique_groups": int(df[column].nunique(dropna=False)),
        "repeated_groups": int(len(repeated_groups)),
        "rows_in_repeated_groups": int(repeated_groups.sum()),
        "repeated_row_rate": float(
            repeated_groups.sum() / len(df)
        ),
        "mean_rows_per_group": float(counts.mean()),
        "max_rows_per_group": int(counts.max()),
        "largest_groups": {
            str(k): int(v)
            for k, v in counts.head(10).items()
        },
    })


@tool
def inspect_feature_target_associations(runtime: ToolRuntime) -> str:
    """Inspect suspicious feature-target associations to help detect possible leakage or post-outcome proxies."""

    dataset_id = runtime.state["dataset_id"]
    target = runtime.state["target"]

    df = load_dataset(dataset_id)

    numeric_columns = (
        df.select_dtypes(include=np.number)
        .columns
        .drop(target, errors="ignore")
    )

    numeric_associations = []

    for column in numeric_columns:
        valid = df[[column, target]].dropna()

        if len(valid) < 2:
            continue

        correlation = valid[column].corr(valid[target])

        if pd.notna(correlation):
            numeric_associations.append({
                "column": column,
                "correlation": float(correlation),
                "abs_correlation": float(abs(correlation)),
            })

    numeric_associations.sort(
        key=lambda x: x["abs_correlation"],
        reverse=True
    )

    return json.dumps({
        "target": target,
        "numeric_associations": numeric_associations,
    })

In [9]:
@tool
def query_dataset_sql(query: str, runtime: ToolRuntime) -> str:
    """Run a read-only DuckDB SELECT query against the current dataset, available as table `data`."""
    normalized = query.strip().lower()
    allowed_start = normalized.startswith('select') or normalized.startswith('with')
    forbidden = re.search(
        r'\b(insert|update|delete|drop|alter|create|copy|attach|detach|pragma|install|load)\b',
        normalized,
    )
    if not allowed_start or forbidden:
        return 'Rejected. Only read-only SELECT/WITH queries against table `data` are allowed.'

    df = load_dataset(runtime.state['dataset_id'])
    con = duckdb.connect(database=':memory:')
    try:
        con.register('data', df)
        result = con.execute(query).df()
        if len(result) > 30:
            result = result.head(30)
        return result.to_json(orient='records', date_format='iso')
    finally:
        con.close()

The tool surface exposes schema, target balance, missingness, duplicates, time candidates, repeated entities, feature-target associations, and read-only DuckDB queries.

`ToolRuntime` injects the active dataset from graph state instead of letting the model choose arbitrary file paths.

## 6. Tool verification

In [10]:
from types import SimpleNamespace


def test_runtime(dataset_id):
    return SimpleNamespace(
        state=get_task_spec(dataset_id)
    )

In [11]:
# Schema: dataset_04
runtime = test_runtime("dataset_04")
print(
    inspect_schema.func(
        runtime=runtime
    )
)

{"shape": {"rows": 3200, "columns": 10}, "columns_info": {"credit_score_signal": {"dtype": "float64", "unique_count": 3200}, "debt_ratio_signal": {"dtype": "float64", "unique_count": 3200}, "income_signal": {"dtype": "float64", "unique_count": 3200}, "utilization_signal": {"dtype": "float64", "unique_count": 3200}, "history_signal": {"dtype": "float64", "unique_count": 3200}, "noise_signal": {"dtype": "float64", "unique_count": 3200}, "default_status_after_30d": {"dtype": "int64", "unique_count": 2}, "collection_amount_after_default": {"dtype": "float64", "unique_count": 588}, "application_id": {"dtype": "object", "unique_count": 3200}, "defaulted": {"dtype": "int64", "unique_count": 2}}, "sample": [{"credit_score_signal": 0.5361185987581178, "debt_ratio_signal": 1.093198513584813, "income_signal": -0.1643083211592484, "utilization_signal": -0.8581704518183669, "history_signal": -0.0311320345756116, "noise_signal": 2.003981532887928, "default_status_after_30d": 0, "collection_amount_af

In [12]:
# Target profile: dataset_01
runtime = test_runtime("dataset_01")
print(
    profile_target.func(
        runtime=runtime
    )
)

{"target_column": "churn", "problem_type": "classification", "target_profile": [{"churn": 0, "count": 3379, "proportion": 0.9654285714285714}, {"churn": 1, "count": 121, "proportion": 0.03457142857142857}]}


In [13]:
# Time candidate: dataset_02.timestamp
runtime = test_runtime("dataset_02")
print(
    inspect_time_candidate.func(
        column="timestamp",
        runtime=runtime
    )
)

{"column": "timestamp", "parseable_rate": 1.0, "min_time": "2024-01-01 00:00:00", "max_time": "2024-05-04 23:00:00", "is_monotonic_increasing": true, "target_over_time": [{"period": 1, "start": "2024-01-01 00:00:00", "end": "2024-02-01 05:00:00", "count": 750, "target_mean": 166.3869233694265}, {"period": 2, "start": "2024-02-01 06:00:00", "end": "2024-03-03 11:00:00", "count": 750, "target_mean": 185.73210434489806}, {"period": 3, "start": "2024-03-03 12:00:00", "end": "2024-04-03 17:00:00", "count": 750, "target_mean": 204.79943448201195}, {"period": 4, "start": "2024-04-03 18:00:00", "end": "2024-05-04 23:00:00", "count": 750, "target_mean": 223.1559775895097}]}


c:\Users\avado\OneDrive\Desktop\DataEngineeringStudy\.venv\Lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [14]:
# Group candidate: dataset_03.machine_id
runtime = test_runtime("dataset_03")
print(
    inspect_group_candidate.func(
        column="machine_id",
        runtime=runtime
    )
)

{"column": "machine_id", "row_count": 2700, "unique_groups": 450, "repeated_groups": 450, "rows_in_repeated_groups": 2700, "repeated_row_rate": 1.0, "mean_rows_per_group": 6.0, "max_rows_per_group": 6, "largest_groups": {"M_0449": 6, "M_0000": 6, "M_0433": 6, "M_0432": 6, "M_0431": 6, "M_0430": 6, "M_0429": 6, "M_0428": 6, "M_0427": 6, "M_0426": 6}}


In [15]:
# Feature-target associations: dataset_04
runtime = test_runtime("dataset_04")
print(
    inspect_feature_target_associations.func(
        runtime=runtime
    )
)

{"target": "defaulted", "numeric_associations": [{"column": "default_status_after_30d", "correlation": 1.0, "abs_correlation": 1.0}, {"column": "collection_amount_after_default", "correlation": 0.8896334357378105, "abs_correlation": 0.8896334357378105}, {"column": "credit_score_signal", "correlation": -0.2220610370045671, "abs_correlation": 0.2220610370045671}, {"column": "debt_ratio_signal", "correlation": -0.2032274743497082, "abs_correlation": 0.2032274743497082}, {"column": "history_signal", "correlation": -0.17725156994370347, "abs_correlation": 0.17725156994370347}, {"column": "income_signal", "correlation": 0.1493442399534995, "abs_correlation": 0.1493442399534995}, {"column": "noise_signal", "correlation": -0.028246391891090766, "abs_correlation": 0.028246391891090766}, {"column": "utilization_signal", "correlation": -0.010215865781157357, "abs_correlation": 0.010215865781157357}]}


The tools expose the intended signals correctly: 3.46% churn prevalence, a strictly ordered timestamp series, six rows per machine, and explicit post-outcome fields with near-perfect target association.

## 7. Diagnostic agent loop

In [16]:
DIAGNOSTIC_SYSTEM_PROMPT = '''
You are an ML methodology investigator.

Inspect an unfamiliar supervised-learning dataset BEFORE any model is trained.

Determine from evidence:
- whether class imbalance affects metric choice;
- whether chronology must be preserved;
- whether repeated entities require grouped validation;
- whether suspicious post-outcome / leakage features exist;
- whether missingness or duplicates matter.

Rules:
1. Use tools instead of guessing.
2. Do not train a model.
3. Do not call every tool mechanically.
4. Once evidence is sufficient, stop calling tools and return a concise summary beginning exactly:

DIAGNOSTICS_COMPLETE:
'''.strip()

DIAGNOSTIC_TOOLS = [
    inspect_schema,
    profile_target,
    inspect_missingness,
    inspect_duplicates,
    inspect_time_candidate,
    inspect_group_candidate,
    inspect_feature_target_associations,
    query_dataset_sql,
]

diagnostic_llm = llm.bind_tools(DIAGNOSTIC_TOOLS)
diagnostic_tool_node = ToolNode(DIAGNOSTIC_TOOLS, handle_tool_errors=True)

In [17]:
def diagnostic_agent_node(state: AgentState):

    task_context = (
        f"Objective: {state['objective']}"
        f"Problem type: {state['problem_type']}"
        f"Target: {state['target']}"
    )

    ai_message = diagnostic_llm.invoke([
        SystemMessage(DIAGNOSTIC_SYSTEM_PROMPT),
        HumanMessage(task_context),
        *state['messages']
    ])

    return {'messages': [ai_message]}


def route_after_diagnostic(state: AgentState) -> Literal['diagnostic_tools', 'plan']:

    last = state["messages"][-1]

    if getattr(last, "tool_calls", None):
        return "diagnostic_tools"

    return 'plan'

The model decides which observation to request next; the graph decides which transitions are allowed. Training is not reachable from the diagnostic node.

## 8. Structured planning and human approval

In [18]:
class ExperimentPlan(BaseModel):
    split_strategy: Literal['random', 'stratified_random', 'chronological', 'grouped']
    primary_metric: Literal['average_precision', 'roc_auc', 'f1', 'accuracy', 'rmse', 'mae', 'r2']
    model_name: Literal[
        'logistic_regression',
        'random_forest_classifier',
        'hist_gradient_boosting_classifier',
        'ridge',
        'random_forest_regressor',
        'hist_gradient_boosting_regressor',
    ]
    time_column: str | None = None
    group_column: str | None = None
    drop_columns: list[str] = Field(default_factory=list)
    risk_flags: list[str] = Field(default_factory=list)
    evidence: list[str] = Field(default_factory=list)
    rationale: str

planner_llm = llm.with_structured_output(ExperimentPlan, method='json_schema')

PLANNER_SYSTEM_PROMPT = '''
Design ONE conservative baseline experiment from diagnostic evidence.

Requirements:
- validation must match intended generalization;
- metric must match objective;
- classification tasks must use classification metrics and classifier models;
- regression tasks must use regression metrics and regressor models;
- stratified_random is valid only for classification targets, never continuous regression targets;
- post-outcome / leakage fields must be dropped;
- entity IDs must not be ordinary predictive features;
- time/group columns used for splitting should normally be removed from model features;
- choose a simple baseline, not tuning;
- never use test information.
'''.strip()

In [19]:
def compact_transcript(messages, max_messages=30):
    parts = []
    for message in messages[-max_messages:]:
        if isinstance(message, ToolMessage):
            parts.append(f'TOOL {message.name}:\n{message.content}')
        elif isinstance(message, AIMessage) and message.content:
            parts.append(f'AGENT:\n{message.content}')
    return '\n\n'.join(parts)


def plan_node(state: AgentState):

    task_context = (
        f"Objective: {state['objective']}\n"
        f"Problem type: {state['problem_type']}\n"
        f"Target: {state['target']}"
    )

    diagnostic_context = compact_transcript(
        state["messages"]
    )

    plan = planner_llm.invoke([
        SystemMessage(content=PLANNER_SYSTEM_PROMPT),
        HumanMessage(content=task_context),
        HumanMessage(content=diagnostic_context),
    ])

    return {
        "plan": plan.model_dump()
    }

In [20]:
class ApprovalDecision(BaseModel):
    approved: bool
    note: str | None = None


def approval_node(state: AgentState):
    decision = interrupt(
        {
            'question': 'Approve this ML experiment plan?',
            'dataset_id': state['dataset_id'],
            'plan': state['plan'],
        },
        response_schema=ApprovalDecision,
    )
    return {
        'human_approved': decision.approved,
        'human_note': decision.note or '',
    }


def route_after_approval(state: AgentState) -> Literal['experiment', 'revise_plan']:
    return 'experiment' if state['human_approved'] else 'revise_plan'

`ExperimentPlan` constrains split strategy, metric, model family, time/group fields, exclusions, risks, evidence, and rationale. LangGraph `interrupt()` creates a genuine external approval boundary before execution.

## 9. Deterministic experiment engine

In [21]:
def build_preprocessor(X_train: pd.DataFrame):
    numeric_columns = X_train.select_dtypes(include=np.number).columns.tolist()
    categorical_columns = [c for c in X_train.columns if c not in numeric_columns]

    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ])

    return ColumnTransformer([
        ('numeric', numeric_pipeline, numeric_columns),
        ('categorical', categorical_pipeline, categorical_columns),
    ])


def build_model(model_name: str):
    models = {
        'logistic_regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE),
        'random_forest_classifier': RandomForestClassifier(n_estimators=200, min_samples_leaf=3, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
        'hist_gradient_boosting_classifier': HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        'ridge': Ridge(),
        'random_forest_regressor': RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1),
        'hist_gradient_boosting_regressor': HistGradientBoostingRegressor(random_state=RANDOM_STATE),
    }
    return clone(models[model_name])

In [22]:
def make_holdout_split(df: pd.DataFrame, target: str, plan: dict[str, Any]):
    strategy = plan['split_strategy']
    y = df[target].copy()
    X = df.drop(columns=[target]).copy()

    if strategy == 'chronological':
        time_column = plan['time_column']
        if not time_column:
            raise ValueError('chronological split requires time_column')
        order = pd.to_datetime(X[time_column]).sort_values().index
        X, y = X.loc[order], y.loc[order]
        split_index = int(len(X) * 0.75)
        train_idx, valid_idx = X.index[:split_index], X.index[split_index:]

    elif strategy == 'grouped':
        group_column = plan['group_column']
        if not group_column:
            raise ValueError('grouped split requires group_column')
        splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
        train_pos, valid_pos = next(splitter.split(X, y, groups=X[group_column]))
        train_idx, valid_idx = X.index[train_pos], X.index[valid_pos]

    else:
        stratify = y if strategy == 'stratified_random' else None
        train_idx, valid_idx = train_test_split(
            X.index,
            test_size=0.25,
            random_state=RANDOM_STATE,
            stratify=stratify,
        )

    return X.loc[train_idx].copy(), X.loc[valid_idx].copy(), y.loc[train_idx].copy(), y.loc[valid_idx].copy()

In [23]:
def score_predictions(metric: str, y_true, y_pred, y_score=None):
    if metric == 'accuracy':
        return accuracy_score(y_true, y_pred)
    if metric == 'f1':
        return f1_score(y_true, y_pred)
    if metric == 'average_precision':
        if y_score is None:
            raise ValueError('average_precision requires scores')
        return average_precision_score(y_true, y_score)
    if metric == 'roc_auc':
        if y_score is None:
            raise ValueError('roc_auc requires scores')
        return roc_auc_score(y_true, y_score)
    if metric == 'rmse':
        return mean_squared_error(y_true, y_pred) ** 0.5
    if metric == 'mae':
        return mean_absolute_error(y_true, y_pred)
    if metric == 'r2':
        return r2_score(y_true, y_pred)
    raise ValueError(f'Unsupported metric: {metric}')

In [24]:
CLASSIFICATION_METRICS = {
    "average_precision",
    "roc_auc",
    "f1",
    "accuracy",
}

REGRESSION_METRICS = {
    "rmse",
    "mae",
    "r2",
}

CLASSIFICATION_MODELS = {
    "logistic_regression",
    "random_forest_classifier",
    "hist_gradient_boosting_classifier",
}

REGRESSION_MODELS = {
    "ridge",
    "random_forest_regressor",
    "hist_gradient_boosting_regressor",
}


def validate_plan_for_task(
    state: AgentState,
) -> list[str]:
    """
    Deterministically validate structural compatibility between
    the LLM-generated plan and the supervised-learning task.

    This does not choose the methodology for the agent. It prevents
    invalid combinations from reaching scikit-learn.
    """
    plan = state["plan"]
    problem_type = state["problem_type"]
    errors = []

    if problem_type == "classification":
        if plan["primary_metric"] not in CLASSIFICATION_METRICS:
            errors.append(
                "Classification task requires a classification metric."
            )

        if plan["model_name"] not in CLASSIFICATION_MODELS:
            errors.append(
                "Classification task requires a classifier model."
            )

    elif problem_type == "regression":
        if plan["primary_metric"] not in REGRESSION_METRICS:
            errors.append(
                "Regression task requires a regression metric."
            )

        if plan["model_name"] not in REGRESSION_MODELS:
            errors.append(
                "Regression task requires a regressor model."
            )

        if plan["split_strategy"] == "stratified_random":
            errors.append(
                "stratified_random is invalid for a continuous "
                "regression target; use random, chronological, "
                "or grouped validation as appropriate."
            )

    else:
        errors.append(
            f"Unsupported problem_type: {problem_type}"
        )

    df = load_dataset(
        state["dataset_id"]
    )

    feature_columns = set(
        df.columns
    ) - {state["target"]}

    if plan["split_strategy"] == "chronological":
        time_column = plan.get(
            "time_column"
        )

        if not time_column:
            errors.append(
                "chronological split requires time_column."
            )
        elif time_column not in feature_columns:
            errors.append(
                f"time_column '{time_column}' is not present "
                "in the dataset features."
            )

    if plan["split_strategy"] == "grouped":
        group_column = plan.get(
            "group_column"
        )

        if not group_column:
            errors.append(
                "grouped split requires group_column."
            )
        elif group_column not in feature_columns:
            errors.append(
                f"group_column '{group_column}' is not present "
                "in the dataset features."
            )

    invalid_drop_columns = [
        column
        for column in plan.get(
            "drop_columns",
            [],
        )
        if (
            column != state["target"]
            and column not in df.columns
        )
    ]

    if invalid_drop_columns:
        errors.append(
            "Unknown drop_columns: "
            + ", ".join(
                invalid_drop_columns
            )
        )

    return errors

In [25]:
def run_experiment(state: AgentState) -> dict[str, Any]:
    dataset_id = state['dataset_id']
    target = state['target']
    problem_type = state['problem_type']
    plan = state['plan']

    df = load_dataset(dataset_id)
    X_train, X_valid, y_train, y_valid = make_holdout_split(df, target, plan)

    drop_columns = set(plan.get('drop_columns', []))
    if plan.get('time_column'):
        drop_columns.add(plan['time_column'])
    if plan.get('group_column'):
        drop_columns.add(plan['group_column'])
    drop_columns = [c for c in drop_columns if c in X_train.columns]

    X_train = X_train.drop(columns=drop_columns)
    X_valid = X_valid.drop(columns=drop_columns)

    pipeline = Pipeline([
        ('preprocessor', build_preprocessor(X_train)),
        ('model', build_model(plan['model_name'])),
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_valid)

    y_score = None
    if problem_type == 'classification':
        if hasattr(pipeline, 'predict_proba'):
            y_score = pipeline.predict_proba(X_valid)[:, 1]
        elif hasattr(pipeline, 'decision_function'):
            y_score = pipeline.decision_function(X_valid)

    metric_value = score_predictions(plan['primary_metric'], y_valid, y_pred, y_score)

    diagnostics = {
        'train_rows': len(X_train),
        'valid_rows': len(X_valid),
        'n_features_before_encoding': X_train.shape[1],
        'dropped_columns': drop_columns,
    }

    if plan['split_strategy'] == 'grouped':
        group_column = plan['group_column']
        train_groups = set(df.loc[X_train.index, group_column])
        valid_groups = set(df.loc[X_valid.index, group_column])
        diagnostics['group_overlap_count'] = len(train_groups & valid_groups)

    return {
        'dataset_id': dataset_id,
        'plan': plan,
        'primary_metric': plan['primary_metric'],
        'primary_metric_value': float(metric_value),
        'split_strategy': plan['split_strategy'],
        'model_name': plan['model_name'],
        'diagnostics': diagnostics,
    }

def experiment_node(state: AgentState):
    """
    Execute one approved experiment attempt.

    Expected plan errors are caught before sklearn runs. Runtime
    exceptions are converted into structured experiment evidence so
    the critic can decide whether replanning can recover from them.
    """
    validation_errors = validate_plan_for_task(
        state
    )

    if validation_errors:
        result = {
            "dataset_id": state["dataset_id"],
            "status": "invalid_plan",
            "plan": state["plan"],
            "validation_errors": validation_errors,
        }

    else:
        try:
            result = run_experiment(
                state
            )
            result["status"] = "success"

        except Exception as exc:
            result = {
                "dataset_id": state["dataset_id"],
                "status": "execution_error",
                "plan": state["plan"],
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }

    print(
        json.dumps(
            result,
            indent=2,
            default=str,
        )
    )

    return {
        "experiments": [result],
        "experiment_iteration": (
            state.get(
                "experiment_iteration",
                0,
            )
            + 1
        ),
    }


Chronological, grouped, stratified, and ordinary random splits are enforced in Python. Preprocessing is fitted inside the scikit-learn pipeline, and metrics are computed deterministically outside the LLM.

## 10. Methodology critic and bounded replanning

In [26]:
class Critique(BaseModel):
    decision: Literal['accept', 'revise']
    issues: list[str] = Field(default_factory=list)
    recommended_changes: list[str] = Field(default_factory=list)
    rationale: str

critic_llm = llm.with_structured_output(Critique, method='json_schema')

CRITIC_SYSTEM_PROMPT = '''
You are a skeptical ML methodology reviewer.

Look for concrete methodological problems:
- validation leakage;
- post-outcome features left in X;
- incorrect time/group splitting;
- inappropriate primary metric;
- suspicious near-perfect performance;
- identifiers used as predictive features;
- contradictions with diagnostic evidence.

Do not demand tuning merely because performance is mediocre.

If the latest experiment has status='invalid_plan':
- return REVISE;
- use the deterministic validation errors to propose a compatible plan.

If the latest experiment has status='execution_error':
- treat the exception type and message as new evidence;
- return REVISE when the error can plausibly be fixed by changing the experiment plan;
- recommend the smallest concrete change that addresses the error;
- do not invent a model result that was never produced.

If the error appears to be an infrastructure or programming failure rather than a methodology/configuration problem, state that clearly in the critique.

Return ACCEPT only when a successful experiment is methodologically defensible.
'''.strip()

In [27]:
def critic_node(state: AgentState):

    task_context = (
        f"Objective: {state['objective']}\n"
        f"Problem type: {state['problem_type']}\n"
        f"Target: {state['target']}"
    )

    diagnostic_context = compact_transcript(
        state["messages"]
    )

    plan = state["plan"]
    latest_experiment_result = state["experiments"][-1]

    critique = critic_llm.invoke([
        SystemMessage(content=CRITIC_SYSTEM_PROMPT),
        HumanMessage(content=task_context),
        HumanMessage(
            content=f"Diagnostic evidence:\n{diagnostic_context}"
        ),
        HumanMessage(
            content=f"Experiment plan:\n{json.dumps(plan, indent=2)}"
        ),
        HumanMessage(
            content=(
                "Latest experiment result:\n"
                f"{json.dumps(latest_experiment_result, indent=2, default=str)}"
            )
        ),
    ])

    return {
        "critique": critique.model_dump()
    }

def route_after_critic(
    state: AgentState,
) -> Literal[
    "report",
    "revise_plan",
]:
    latest_experiment = (
        state.get(
            "experiments",
            [],
        )[-1]
    )

    status = latest_experiment.get(
        "status",
        "success",
    )

    iterations = state.get(
        "experiment_iteration",
        0,
    )

    # An invalid or failed execution should not silently terminate
    # the workflow while another bounded recovery attempt remains.
    if status != "success":
        if iterations < MAX_EXPERIMENT_ITERATIONS:
            return "revise_plan"
        return "report"

    if (
        state["critique"]["decision"]
        == "revise"
        and iterations < MAX_EXPERIMENT_ITERATIONS
    ):
        return "revise_plan"

    return "report"


In [28]:
REPLAN_SYSTEM_PROMPT = '''
Revise the current ExperimentPlan using concrete human or critic feedback.
Preserve correct decisions. Change only what evidence justifies.
If the previous experiment failed to execute, use the error type/message and critic feedback to repair the plan rather than repeating it unchanged.
Do not introduce hyperparameter tuning.
'''.strip()


def revise_plan_node(state: AgentState):

    diagnostic_context = compact_transcript(
        state["messages"]
    )

    prior_plan = state["plan"]
    human_note = state.get("human_note", "")
    critique = state.get("critique", {})

    revised_plan = planner_llm.invoke([
        SystemMessage(content=REPLAN_SYSTEM_PROMPT),

        HumanMessage(
            content=(
                f"Objective: {state['objective']}\n"
                f"Problem type: {state['problem_type']}\n"
                f"Target: {state['target']}"
            )
        ),

        HumanMessage(
            content=(
                "Diagnostic evidence:\n"
                f"{diagnostic_context}"
            )
        ),

        HumanMessage(
            content=(
                "Prior experiment plan:\n"
                f"{json.dumps(prior_plan, indent=2)}"
            )
        ),

        HumanMessage(
            content=(
                "Human feedback:\n"
                f"{human_note or 'No human feedback provided.'}"
            )
        ),

        HumanMessage(
            content=(
                "Latest experiment attempt:\n"
                f"{json.dumps(state.get('experiments', [])[-1] if state.get('experiments') else {}, indent=2, default=str)}"
            )
        ),

        HumanMessage(
            content=(
                "Critic feedback:\n"
                f"{json.dumps(critique, indent=2, default=str)}"
            )
        ),
    ])

    return {
        "plan": revised_plan.model_dump(),
        "human_approved": False,
    }

The critic is methodological rather than performance-seeking. Revision requires a concrete problem such as leakage, invalid validation, target-informed feature selection, or an inappropriate metric. The loop is capped by `MAX_EXPERIMENT_ITERATIONS`.

## 11. Reporting and graph topology

In [29]:
REPORT_SYSTEM_PROMPT = '''
Write a concise ML investigation report using only supplied state and experiment evidence.
Include:
- problem formulation;
- risks discovered;
- final validation design;
- primary metric and why;
- excluded columns and why;
- experiment result;
- revision history;
- limitations;
- recommended next experiment.
If no experiment completed successfully, explicitly report the unresolved execution failure instead of inventing a metric result.
Do not claim production readiness.
'''.strip()


def report_node(state: AgentState):

    report_context = (
        f"Objective:\n{state['objective']}\n\n"
        f"Problem type:\n{state['problem_type']}\n\n"
        f"Target:\n{state['target']}\n\n"
        f"Diagnostic evidence:\n"
        f"{compact_transcript(state['messages'])}\n\n"
        f"Final plan:\n"
        f"{json.dumps(state['plan'], indent=2, default=str)}\n\n"
        f"Human feedback:\n"
        f"{state.get('human_note', '')}\n\n"
        f"Experiment history:\n"
        f"{json.dumps(state.get('experiments', []), indent=2, default=str)}\n\n"
        f"Latest critique:\n"
        f"{json.dumps(state.get('critique', {}), indent=2, default=str)}"
    )

    response = llm.invoke([
        SystemMessage(content=REPORT_SYSTEM_PROMPT),
        HumanMessage(content=report_context),
    ])

    if isinstance(response.content, str):
        final_report = response.content
    else:
        final_report = "\n".join(
            block["text"]
            for block in response.content
            if isinstance(block, dict) and block.get("type") == "text"
        )

    return {
        "final_report": final_report
    }

In [30]:
def build_ml_investigator_graph():
    builder = StateGraph(AgentState)

    # Add Nodes
    builder.add_node("diagnostic_agent", diagnostic_agent_node)
    builder.add_node("diagnostic_tools", diagnostic_tool_node)
    builder.add_node("plan", plan_node)
    builder.add_node("approval", approval_node)
    builder.add_node("experiment", experiment_node)
    builder.add_node("critic", critic_node)
    builder.add_node("revise_plan", revise_plan_node)
    builder.add_node("report", report_node)

    # Wire Graph
    builder.add_edge(START, "diagnostic_agent")
    builder.add_conditional_edges("diagnostic_agent", route_after_diagnostic, ["diagnostic_tools", "plan"])
    builder.add_edge("diagnostic_tools", "diagnostic_agent")
    builder.add_edge("plan", "approval")
    builder.add_conditional_edges("approval", route_after_approval, ["experiment", "revise_plan"])
    builder.add_edge("experiment", "critic")
    builder.add_conditional_edges("critic", route_after_critic, ["report", "revise_plan"])
    builder.add_edge("revise_plan", "approval")
    builder.add_edge("report", END)

    graph = builder.compile(checkpointer=InMemorySaver())

    return graph

## 12. Interactive leakage case

In [31]:
ml_investigator = build_ml_investigator_graph()

CASE_DATASET = "dataset_04"
task = get_task_spec(CASE_DATASET)

initial_state = {
    **task,
    'messages': [
        HumanMessage(
            content=(
                f"Investigate dataset '{CASE_DATASET}' and design a defensible baseline experiment.\n\n"
                f"Objective: {task['objective']}\n"
                f"Target: {task['target']}\n"
                f"Problem type: {task['problem_type']}"
            )
        )
    ],
    'experiments': [],
    'experiment_iteration': 0,
}

thread_id = f'interactive-{uuid.uuid4()}'
config = {
    'configurable': {'thread_id': thread_id},
    'recursion_limit': 80,
}

result = ml_investigator.invoke(initial_state, config=config)
result

{'dataset_id': 'dataset_04',
 'target': 'defaulted',
 'problem_type': 'classification',
 'objective': 'Predict loan default at application time. Only information available at application time is valid.',
 'messages': [HumanMessage(content="Investigate dataset 'dataset_04' and design a defensible baseline experiment.\n\nObjective: Predict loan default at application time. Only information available at application time is valid.\nTarget: defaulted\nProblem type: classification", additional_kwargs={}, response_metadata={}, id='fa291cdf-6e52-46b6-9a0d-61401f835d08'),
  AIMessage(content=[{'id': 'rs_010c6be15fdce013006ad5cb594aac87d2bab89033bfabd81e', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABq1ctaVhv0_CroD53rnTWCLqLBXmn6_3jkS7iS-aymaoKgzUP2xBDwzasnF0SM08BsWZqOc_XrnGSvPSJUSupmgBelfFK8MBtE-YDtI1s6c2O9XerBS7cJIKPEzN0TdJDvMYxGIb3kXhhWxuYaOZgLe_Bd-xR5zYS6v9W-k_ieiKAJ8uNpE4GHxogJ5vIzIbcvcR-v8bFZa4XW2KkGe5SynX2eu3MiOrebtAWEUHPHw10xAI89n9vPPi2Zf7d7kY5Ee8K9PfAe

In [32]:
# The graph should pause at human approval.
result.get('__interrupt__')

[Interrupt(value={'question': 'Approve this ML experiment plan?', 'dataset_id': 'dataset_04', 'plan': {'split_strategy': 'chronological', 'primary_metric': 'average_precision', 'model_name': 'logistic_regression', 'time_column': 'application_id', 'group_column': None, 'drop_columns': ['default_status_after_30d', 'collection_amount_after_default', 'application_id'], 'risk_flags': ['default_status_after_30d is a perfect post-outcome leakage field and must be excluded', 'collection_amount_after_default is observed after default and must be excluded', 'application_id is a unique entity identifier and must not be used as a feature', 'No explicit timestamp is available; application_id order is used only as a conservative arrival-order proxy', 'The positive class is imbalanced at 18.34%, so accuracy is not an appropriate primary metric'], 'evidence': ['The target is binary classification with 587 defaults out of 3,200 applications (18.34%).', 'default_status_after_30d has correlation 1.0 with

In [33]:
# Inspect the proposed plan before editing this decision.
approval = {
    "approved": True,
    "note": "Looks Good.",
}
result = ml_investigator.invoke(
    Command(resume=approval),
    config=config,
)

result

C:\Users\avado\AppData\Local\Temp\ipykernel_4960\971448968.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  order = pd.to_datetime(X[time_column]).sort_values().index


{
  "dataset_id": "dataset_04",
  "status": "execution_error",
  "plan": {
    "split_strategy": "chronological",
    "primary_metric": "average_precision",
    "model_name": "logistic_regression",
    "time_column": "application_id",
    "group_column": null,
    "drop_columns": [
      "default_status_after_30d",
      "collection_amount_after_default",
      "application_id"
    ],
    "risk_flags": [
      "default_status_after_30d is a perfect post-outcome leakage field and must be excluded",
      "collection_amount_after_default is observed after default and must be excluded",
      "application_id is a unique entity identifier and must not be used as a feature",
      "No explicit timestamp is available; application_id order is used only as a conservative arrival-order proxy",
      "The positive class is imbalanced at 18.34%, so accuracy is not an appropriate primary metric"
    ],
    "evidence": [
      "The target is binary classification with 587 defaults out of 3,200 appl

{'dataset_id': 'dataset_04',
 'target': 'defaulted',
 'problem_type': 'classification',
 'objective': 'Predict loan default at application time. Only information available at application time is valid.',
 'messages': [HumanMessage(content="Investigate dataset 'dataset_04' and design a defensible baseline experiment.\n\nObjective: Predict loan default at application time. Only information available at application time is valid.\nTarget: defaulted\nProblem type: classification", additional_kwargs={}, response_metadata={}, id='fa291cdf-6e52-46b6-9a0d-61401f835d08'),
  AIMessage(content=[{'id': 'rs_010c6be15fdce013006ad5cb594aac87d2bab89033bfabd81e', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABq1ctaVhv0_CroD53rnTWCLqLBXmn6_3jkS7iS-aymaoKgzUP2xBDwzasnF0SM08BsWZqOc_XrnGSvPSJUSupmgBelfFK8MBtE-YDtI1s6c2O9XerBS7cJIKPEzN0TdJDvMYxGIb3kXhhWxuYaOZgLe_Bd-xR5zYS6v9W-k_ieiKAJ8uNpE4GHxogJ5vIzIbcvcR-v8bFZa4XW2KkGe5SynX2eu3MiOrebtAWEUHPHw10xAI89n9vPPi2Zf7d7kY5Ee8K9PfAe

In [34]:
# result.get('__interrupt__')

# OPTIONAL if a second approval is requested:
# result = ml_investigator.invoke(
#     Command(resume={'approved': True, 'note': 'Approved revised plan.'}),
#     config=config,
# )
print(result.get('final_report', ''))

### Execution-error recovery

The interactive loan-default run produced a useful failure case.

The planner inferred that `application_id` could act as an arrival-order proxy and proposed a chronological split. The human-approved experiment then attempted to parse:

```text
APP_00000
APP_00001
...
```

as datetimes and raised a `DateParseError`.

The important behavior is that the graph **did not crash**. `experiment_node()` converted the exception into structured state:

```text
status = execution_error
error_type = DateParseError
error_message = ...
```

This result becomes new evidence for the critic/replanning path rather than terminating the workflow.

Because this notebook demonstrates a real human approval boundary, the interactive trace pauses again when a revised plan requires another approval. The repeated benchmark later uses an auto-approval harness so the full agent can be evaluated without manual intervention.


## 13. Tool trace

In [35]:
def summarize_trace(state: AgentState):
    rows = []
    for message in state.get('messages', []):
        if isinstance(message, AIMessage):
            for call in message.tool_calls or []:
                rows.append({'type': 'tool_call', 'name': call['name'], 'args': call['args']})
        elif isinstance(message, ToolMessage):
            rows.append({'type': 'tool_result', 'name': message.name, 'preview': str(message.content)[:200]})
    return pd.DataFrame(rows)

# After reaching END:
trace_df = summarize_trace(result)
display(trace_df)

,type,name,args,preview
0,tool_call,inspect_schema,{},NaN
1,tool_call,profile_target,{},NaN
2,tool_call,inspect_missingness,{},NaN
3,tool_call,inspect_duplicates,{},NaN
4,tool_result,inspect_schema,NaN,"{""shape"": {""rows"": 3200, ""columns"": 10}, ""colu..."
5,tool_result,profile_target,NaN,"{""target_column"": ""defaulted"", ""problem_type"":..."
6,tool_result,inspect_missingness,NaN,"{""missingness"": [{""column"": ""credit_score_sign..."
7,tool_result,inspect_duplicates,NaN,"{""duplicate_count"": 0, ""duplicate_rate"": 0.0}"
8,tool_call,inspect_feature_target_associations,{},NaN
9,tool_call,inspect_group_candidate,{'column': 'application_id'},NaN


Across the five benchmark tasks, the tool-using agent averages **7.8 diagnostic tool calls per task** and **1.8 experiment attempts per task**.

Two cases reach the maximum of three experiment attempts because the critic continues requesting conservative revisions even after a valid experiment succeeds. This is a useful limitation: critique can become overly cautious when it asks for evidence or transformations that the current tool surface cannot actually provide.


## 14. Agent benchmark

In [36]:
BENCHMARK_EXPECTATIONS = {
    "dataset_01": {
        "accepted_splits": {"stratified_random"},
        "accepted_metrics": {"average_precision"},
        "required_risk_concepts": {"imbalance"},
        "required_drop_columns": {"customer_id"},
    },

    "dataset_02": {
        "accepted_splits": {"chronological"},
        "accepted_metrics": {"rmse", "mae"},
        "required_risk_concepts": {"time"},
        "required_drop_columns": {"timestamp"},
    },

    "dataset_03": {
        "accepted_splits": {"grouped"},
        "accepted_metrics": {
            "roc_auc",
            "average_precision",
            "f1",
        },
        "required_risk_concepts": {"group"},
        "required_drop_columns": {"machine_id"},
    },

    "dataset_04": {
        "accepted_splits": {"stratified_random"},
        "accepted_metrics": {
            "roc_auc",
            "average_precision",
        },
        "required_risk_concepts": {"leak"},
        "required_drop_columns": {
            "default_status_after_30d",
            "collection_amount_after_default",
            "application_id",
        },
    },

    "dataset_05": {
        "accepted_splits": {"random"},
        "accepted_metrics": {
            "rmse",
            "mae",
            "r2",
        },
        "required_risk_concepts": set(),
        "required_drop_columns": set(),
    },
}

In [37]:
def auto_approval_node(state: AgentState):
    return {
        'human_approved': True,
        'human_note': 'Auto-approved by benchmark harness.',
    }


def build_benchmark_graph():
    builder = StateGraph(AgentState)

    # Add Nodes
    builder.add_node("diagnostic_agent", diagnostic_agent_node)
    builder.add_node("diagnostic_tools", diagnostic_tool_node)
    builder.add_node("plan", plan_node)
    builder.add_node("approval", auto_approval_node)
    builder.add_node("experiment", experiment_node)
    builder.add_node("critic", critic_node)
    builder.add_node("revise_plan", revise_plan_node)
    builder.add_node("report", report_node)

    # Wire Graph
    builder.add_edge(START, "diagnostic_agent")
    builder.add_conditional_edges("diagnostic_agent", route_after_diagnostic, ["diagnostic_tools", "plan"])
    builder.add_edge("diagnostic_tools", "diagnostic_agent")
    builder.add_edge("plan", "approval")
    builder.add_conditional_edges("approval", route_after_approval, ["experiment", "revise_plan"])
    builder.add_edge("experiment", "critic")
    builder.add_conditional_edges("critic", route_after_critic, ["report", "revise_plan"])
    builder.add_edge("revise_plan", "approval")
    builder.add_edge("report", END)

    graph = builder.compile(checkpointer=InMemorySaver())

    return graph

In [38]:
def normalize_metric_name(metric: str) -> str:
    text = metric.strip().lower()

    normalized = (
        text.replace("-", " ")
        .replace("_", " ")
        .replace("(", " ")
        .replace(")", " ")
    )

    normalized = " ".join(
        normalized.split()
    )

    if (
        "average precision" in normalized
        or "pr auc" in normalized
        or "prauc" in normalized
    ):
        return "average_precision"

    if (
        "roc auc" in normalized
        or "rocauc" in normalized
    ):
        return "roc_auc"

    if (
        normalized == "mae"
        or "mean absolute error" in normalized
    ):
        return "mae"

    if (
        normalized == "rmse"
        or "root mean squared error" in normalized
    ):
        return "rmse"

    if normalized in {
        "r2",
        "r squared",
        "r^2",
    }:
        return "r2"

    if "accuracy" in normalized:
        return "accuracy"

    if (
        normalized == "f1"
        or "f1 score" in normalized
    ):
        return "f1"

    return text


RISK_PATTERNS = {
    "imbalance": (
        "imbalance",
        "imbalanced",
        "minority",
        "rare positive",
        "class prevalence",
        "positive class",
    ),
    "time": (
        "time",
        "temporal",
        "chronolog",
        "future timestamp",
        "future period",
        "drift",
    ),
    "group": (
        "group",
        "grouped",
        "repeated",
        "same machine",
        "machine",
        "row-wise",
        "unseen machine",
    ),
    "leak": (
        "leak",
        "post-outcome",
        "post outcome",
        "post-default",
        "post default",
        "after_30d",
        "after 30d",
        "after default",
    ),
}


def semantic_risk_detection(
    dataset_id: str,
    decision: dict,
) -> bool:
    expected = BENCHMARK_EXPECTATIONS[
        dataset_id
    ]

    concepts = expected[
        "required_risk_concepts"
    ]

    if not concepts:
        return True

    text_parts = []

    for key in [
        "risk_flags",
        "evidence",
    ]:
        value = decision.get(
            key,
            [],
        )

        if isinstance(value, list):
            text_parts.extend(
                map(str, value)
            )

    text_parts.append(
        str(
            decision.get(
                "rationale",
                "",
            )
        )
    )

    text = " ".join(
        text_parts
    ).lower()

    for concept in concepts:
        patterns = RISK_PATTERNS.get(
            concept,
            (concept,),
        )

        if not any(
            pattern in text
            for pattern in patterns
        ):
            return False

    return True


def score_methodology_decision(
    dataset_id: str,
    decision: dict,
) -> dict:
    expected = BENCHMARK_EXPECTATIONS[
        dataset_id
    ]

    split_ok = (
        decision["split_strategy"]
        in expected["accepted_splits"]
    )

    metric_ok = (
        normalize_metric_name(
            decision["primary_metric"]
        )
        in expected["accepted_metrics"]
    )

    drops_ok = (
        expected["required_drop_columns"]
        <= set(
            decision.get(
                "drop_columns",
                [],
            )
        )
    )

    risk_ok = semantic_risk_detection(
        dataset_id,
        decision,
    )

    return {
        "dataset_id": dataset_id,
        "split_ok": split_ok,
        "metric_ok": metric_ok,
        "required_drops_ok": drops_ok,
        "risk_detection_ok": risk_ok,
        "methodology_success": (
            split_ok
            and metric_ok
            and drops_ok
            and risk_ok
        ),
    }


def evaluate_agent_decision(
    dataset_id: str,
    final_state: AgentState,
) -> dict:
    plan = final_state["plan"]

    scored = score_methodology_decision(
        dataset_id,
        plan,
    )

    trace = summarize_trace(
        final_state
    )

    tool_calls = (
        int(
            (
                trace["type"]
                == "tool_call"
            ).sum()
        )
        if not trace.empty
        else 0
    )

    return {
        **scored,
        "tool_calls": tool_calls,
        "experiments": len(
            final_state.get(
                "experiments",
                [],
            )
        ),
        "final_split": plan[
            "split_strategy"
        ],
        "final_metric": normalize_metric_name(
            plan[
                "primary_metric"
            ]
        ),
        "drop_columns": plan.get(
            "drop_columns",
            [],
        ),
        "risk_flags": plan.get(
            "risk_flags",
            [],
        ),
    }

In [39]:
benchmark_graph = build_benchmark_graph()
benchmark_states = {}
benchmark_rows = []

for dataset_id in DATASETS:
    task = get_task_spec(dataset_id)
    state = {
        **task,
        'messages': [HumanMessage(content=(
            f"Investigate dataset '{dataset_id}'. "
            f"Objective: {task['objective']} Target: {task['target']}."
        ))],
        'experiments': [],
        'experiment_iteration': 0,
    }
    benchmark_config = {
        'configurable': {'thread_id': f'benchmark-{dataset_id}-{uuid.uuid4()}'},
        'recursion_limit': 80,
    }
    final_state = benchmark_graph.invoke(state, config=benchmark_config)
    benchmark_states[dataset_id] = final_state
    benchmark_rows.append(evaluate_agent_decision(dataset_id, final_state))

agent_benchmark_df = pd.DataFrame(benchmark_rows)

{
  "dataset_id": "dataset_01",
  "plan": {
    "split_strategy": "stratified_random",
    "primary_metric": "average_precision",
    "model_name": "logistic_regression",
    "time_column": null,
    "group_column": null,
    "drop_columns": [
      "customer_id"
    ],
    "risk_flags": [
      "Severe class imbalance: only 3.46% of customers churned; accuracy is not an appropriate primary metric.",
      "No observation-time column is available, so future-customer temporal drift cannot be directly validated.",
      "Confirm that anonymized features, especially feature_7 and feature_6, are available before churn occurs and are not post-outcome measurements.",
      "Use class-imbalance-aware reporting, including positive-class precision and recall; do not tune the decision threshold on test data."
    ],
    "evidence": [
      "The target is binary classification with 121 positive cases and 3,379 negative cases.",
      "There are 3,500 unique customer_id values, with no repeated en

c:\Users\avado\OneDrive\Desktop\DataEngineeringStudy\.venv\Lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


{
  "dataset_id": "dataset_02",
  "plan": {
    "split_strategy": "chronological",
    "primary_metric": "mae",
    "model_name": "ridge",
    "time_column": "timestamp",
    "group_column": null,
    "drop_columns": [
      "timestamp"
    ],
    "risk_flags": [
      "Confirm that temperature and promo are known or forecastable before each forecast timestamp; otherwise they may cause availability leakage.",
      "The target mean rises substantially over time, so this simple baseline may underrepresent nonstationary trend.",
      "Do not use future demand or statistics computed from the validation period during training or preprocessing."
    ],
    "evidence": [
      "The objective is to predict demand at future timestamps, requiring temporal rather than random validation.",
      "timestamp is fully parseable, strictly increasing, and spans 2024-01-01 through 2024-05-04 at hourly resolution.",
      "Demand mean increases across chronological periods from approximately 166.4 to 2

C:\Users\avado\AppData\Local\Temp\ipykernel_4960\3205654593.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[column], errors="coerce")
c:\Users\avado\OneDrive\Desktop\DataEngineeringStudy\.venv\Lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


{
  "dataset_id": "dataset_04",
  "plan": {
    "split_strategy": "stratified_random",
    "primary_metric": "average_precision",
    "model_name": "logistic_regression",
    "time_column": null,
    "group_column": null,
    "drop_columns": [
      "default_status_after_30d",
      "collection_amount_after_default",
      "application_id"
    ],
    "risk_flags": [
      "The remaining signal features must be confirmed to be available at application time; otherwise they may introduce leakage.",
      "The positive class is imbalanced: defaulted=1 represents 18.3% of rows, so accuracy is not an appropriate primary metric.",
      "No timestamp or repeated entity group is available; random stratification assumes rows are independently exchangeable."
    ],
    "evidence": [
      "The target is binary classification with 587 positives out of 3,200 rows (18.3%).",
      "default_status_after_30d has perfect correlation with defaulted and is explicitly post-application.",
      "collectio

In [40]:
agent_methodology_metrics = {
    "methodology_success": (
        agent_benchmark_df[
            "methodology_success"
        ].mean()
    ),
    "split_selection": (
        agent_benchmark_df[
            "split_ok"
        ].mean()
    ),
    "metric_selection": (
        agent_benchmark_df[
            "metric_ok"
        ].mean()
    ),
    "critical_risk_detection": (
        agent_benchmark_df[
            "risk_detection_ok"
        ].mean()
    ),
    "required_exclusions": (
        agent_benchmark_df[
            "required_drops_ok"
        ].mean()
    ),
}

print("Tool-using agent benchmark:")
display(agent_benchmark_df)

print("\nAggregate methodology metrics:")
display(
    pd.Series(
        agent_methodology_metrics,
        name="tool_using_agent",
    )
)

Tool-using agent benchmark:


,dataset_id,split_ok,metric_ok,required_drops_ok,risk_detection_ok,methodology_success,tool_calls,experiments,final_split,final_metric,drop_columns,risk_flags
0,dataset_01,True,True,True,True,True,9,1,stratified_random,average_precision,[customer_id],[Severe class imbalance: only 3.46% of custome...
1,dataset_02,True,True,True,True,True,8,3,chronological,mae,"[timestamp, temperature, promo]",[Verify and log the final model feature column...
2,dataset_03,True,True,True,True,True,9,1,grouped,average_precision,"[machine_id, failure]",[All 450 machines have repeated rows (6 each);...
3,dataset_04,True,True,True,True,True,7,3,stratified_random,average_precision,"[default_status_after_30d, collection_amount_a...",[Application-time lineage for credit_score_sig...
4,dataset_05,True,True,True,True,True,6,1,random,rmse,[target],[`segment` has only three repeated categories;...



Aggregate methodology metrics:


methodology_success        1.0
split_selection            1.0
metric_selection           1.0
critical_risk_detection    1.0
required_exclusions        1.0
Name: tool_using_agent, dtype: float64

### Benchmark-scoring correction

The first scorer had two evaluation bugs:

1. risk detection searched for literal words such as `"group"` or `"leak"`, so semantically correct descriptions were marked false;
2. the schema-only baseline was penalized for formatting differences such as `MAE` vs. `mae` and `Average Precision (PR-AUC)` vs. `average_precision`.

These were scorer errors, not agent errors.

### Tool-using agent result

This execution completes all five benchmark tasks successfully.

| Dataset | Final split | Final metric | Tool calls | Experiment attempts | Methodology |
|---|---|---:|---:|---:|---|
| Churn | stratified random | Average Precision | 9 | 1 | Correct |
| Demand | chronological | MAE | 8 | 3 | Correct |
| Machine failure | grouped | Average Precision | 9 | 1 | Correct |
| Loan default | stratified random | Average Precision | 7 | 3 | Correct |
| IID regression | random | RMSE | 6 | 1 | Correct |

Aggregate methodology metrics:

- methodology success: **1.00**
- split selection: **1.00**
- metric selection: **1.00**
- critical-risk detection: **1.00**
- required exclusions: **1.00**

The benchmark also verifies zero group overlap for the grouped machine-failure task.


## 15. Schema-only LLM baseline

In [41]:
class MethodologyAnswer(BaseModel):
    split_strategy: Literal[
        "random",
        "stratified_random",
        "chronological",
        "grouped"
    ]
    primary_metric: str
    drop_columns: list[str]
    risk_flags: list[str]
    rationale: str


llm_only_methodologist = llm.with_structured_output(
    MethodologyAnswer,
    method="json_schema"
)


def llm_only_decision(dataset_id: str):

    task = get_task_spec(dataset_id)
    df = load_dataset(dataset_id)

    schema = {
        column: str(dtype)
        for column, dtype in df.dtypes.items()
    }

    context = (
        f"Objective: {task['objective']}\n"
        f"Target: {task['target']}\n"
        f"Problem type: {task['problem_type']}\n\n"
        f"Columns and dtypes:\n"
        f"{json.dumps(schema, indent=2)}"
    )

    decision = llm_only_methodologist.invoke([
        SystemMessage(
            content=(
                "Design one conservative baseline ML methodology using only "
                "the information provided. Do not assume access to dataset "
                "values or diagnostics that are not shown. Choose an appropriate "
                "validation strategy and primary metric, identify columns that "
                "should be excluded, and state relevant methodological risks."
            )
        ),
        HumanMessage(content=context),
    ])

    return decision.model_dump()

In [42]:
llm_only_rows = []
llm_only_decisions = {}

for dataset_id in DATASETS:
    decision = llm_only_decision(
        dataset_id
    )

    llm_only_decisions[
        dataset_id
    ] = decision

    scored = score_methodology_decision(
        dataset_id,
        decision,
    )

    llm_only_rows.append({
        **scored,
        "final_split": decision[
            "split_strategy"
        ],
        "final_metric": (
            normalize_metric_name(
                decision[
                    "primary_metric"
                ]
            )
        ),
        "drop_columns": decision.get(
            "drop_columns",
            [],
        ),
        "risk_flags": decision.get(
            "risk_flags",
            [],
        ),
    })


llm_only_benchmark_df = pd.DataFrame(
    llm_only_rows
)

llm_only_metrics = {
    "methodology_success": (
        llm_only_benchmark_df[
            "methodology_success"
        ].mean()
    ),
    "split_selection": (
        llm_only_benchmark_df[
            "split_ok"
        ].mean()
    ),
    "metric_selection": (
        llm_only_benchmark_df[
            "metric_ok"
        ].mean()
    ),
    "critical_risk_detection": (
        llm_only_benchmark_df[
            "risk_detection_ok"
        ].mean()
    ),
    "required_exclusions": (
        llm_only_benchmark_df[
            "required_drops_ok"
        ].mean()
    ),
}

comparison_df = pd.DataFrame({
    "Schema-only LLM": (
        llm_only_metrics
    ),
    "Tool-using agent": (
        agent_methodology_metrics
    ),
})

print("Schema-only LLM benchmark:")
display(
    llm_only_benchmark_df
)

print("\nComparison:")
display(
    comparison_df
)

Schema-only LLM benchmark:


,dataset_id,split_ok,metric_ok,required_drops_ok,risk_detection_ok,methodology_success,final_split,final_metric,drop_columns,risk_flags
0,dataset_01,True,True,True,True,True,stratified_random,average_precision,"[customer_id, churn]","[No time or snapshot column is provided, so a ..."
1,dataset_02,True,True,True,True,True,chronological,mae,"[demand, timestamp]",[Random splitting would leak temporal informat...
2,dataset_03,True,True,True,True,True,grouped,average_precision,[machine_id],[Failure prevalence is unspecified; PR AUC may...
3,dataset_04,True,True,True,True,True,stratified_random,average_precision,"[default_status_after_30d, collection_amount_a...",[The two columns explicitly measured after 30 ...
4,dataset_05,True,True,True,True,True,random,rmse,[target],[The IID assumption should be verified; random...



Comparison:


,Schema-only LLM,Tool-using agent
methodology_success,1.0,1.0
split_selection,1.0,1.0
metric_selection,1.0,1.0
critical_risk_detection,1.0,1.0
required_exclusions,1.0,1.0


The schema-only baseline receives only the task objective, target, problem type, column names, and dtypes. It does not receive target prevalence, repeated-group counts, temporal diagnostics, correlations, or SQL results.

In this execution it also scores **5 / 5** on the benchmark:

| Metric | Schema-only LLM | Tool-using agent |
|---|---:|---:|
| Methodology success | **1.00** | **1.00** |
| Split selection | **1.00** | **1.00** |
| Metric selection | **1.00** | **1.00** |
| Critical-risk detection | **1.00** | **1.00** |
| Required exclusions | **1.00** | **1.00** |

This is an important result rather than a failure of the project. The benchmark tasks are intentionally interpretable, and several constraints are already visible in names such as `timestamp`, `machine_id`, and post-outcome fields.

The agent's demonstrated value is therefore **not superior headline accuracy on this small benchmark**. Its value is:

- empirical verification through tools;
- explicit, inspectable state;
- deterministic execution boundaries;
- human approval before actions;
- structured error capture;
- bounded critic/replan behavior.


## 16. Baseline experiment outcomes

| Dataset | Validation design | Metric | Final result |
|---|---|---|---:|
| Churn | stratified random | Average Precision | 0.2733 |
| Demand | chronological | MAE | 37.7074 |
| Machine failure | grouped | Average Precision | 0.5294 |
| Loan default | stratified random | Average Precision | 0.3646 |
| IID regression | random | RMSE | 17.4004 |

These values are not compared across datasets. Their purpose is to show that each final methodology can be executed under the intended validation regime.


## 17. Main conclusions

- Constrained tools provide **evidence acquisition**, not unrestricted code execution.
- Explicit LangGraph state separates observations, plans, approvals, experiments, errors, critique, and reporting.
- Human approval is a real external control boundary.
- Runtime failures can be converted into **agent observations** instead of crashing the workflow.
- Deterministic Python remains responsible for splitting, preprocessing, fitting, and metrics.
- On this small benchmark, the schema-only LLM and tool-using agent both reach **5 / 5 methodology success**.
- The strongest advantage of the agent is therefore **auditability, evidence, recovery, and control**, not higher benchmark accuracy.
- The critic can also over-revise: two tasks use all three experiment attempts despite already having a valid successful experiment.
- Agent evaluation needs the same discipline as ML evaluation; literal scoring rules and stochastic runs can otherwise create misleading conclusions.


## 18. Limitations

1. **Only five benchmark tasks.** A 5/5 result is descriptive rather than a reliable estimate of agent accuracy.

2. **Highly interpretable schemas.** Column names and task descriptions expose several methodological constraints, which helps the schema-only LLM reach the same 5/5 score.

3. **Single stochastic run.** Tool traces, plans, and critic behavior can vary between executions.

4. **Overactive critic behavior.** The demand and default tasks use three experiment attempts because the critic requests additional conservative changes even after successful runs.

5. **Limited action surface.** The agent cannot perform arbitrary feature engineering or resolve all provenance questions raised by the critic.

6. **Human approval is demonstrated, not benchmarked.** Repeated evaluation replaces the approval node with auto-approval.

7. **Synthetic benchmark ground truth is hand-authored.** Multiple methodologies may be defensible in real applications.

8. **No latency / cost optimization.** The project records tool-call and experiment counts but does not optimize runtime economics.


## 19. Final architecture

```text
Task + dataset
      ↓
GPT-5.6 Luna diagnostic agent
      ↓
read-only tools
      ↓
Pydantic ExperimentPlan
      ↓
LangGraph interrupt
      ↓
human approval
      ↓
deterministic sklearn experiment
      ↓
methodology critic
      ├── accept → final report
      └── revise → approval → experiment
```